# Brainstorming and Focus Group Quantitative Experimentation 2.1:**Difficult people** under **action correction** + **divergence intervention**

Can we use TinyTroupe to brainstorm product ideas?

In [1]:
import sys

from pprint import pprint

from tinytroupe import config_manager
from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.experimentation import InPlaceExperimentRunner
from tinytroupe.steering import Intervention
from tinytroupe.examples import *
from tinytroupe.validation import propositions
from tinytroupe.extraction import ResultsExtractor
from tinytroupe.utils.parallel import parallel_map_dict, parallel_map_cross
from tinytroupe.validation import hard_persona_adherence, persona_adherence, self_consistency, fluency, task_completion, divergence

# specific utilities
from common_utils import *


!!!!
DISCLAIMER: TinyTroupe relies on Artificial Intelligence (AI) models to generate content. 
The AI models are not perfect and may produce inappropriate or inaccurate results. 
For any serious or consequential use, please review the generated content before using it.
!!!!

Looking for default config on: C:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\tinytroupe\utils\..\config.ini
Found custom config on: c:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\publications\paper_artifacts_april-2026\config.ini
TinyTroupe version: 0.8.0
Current date and time (local): 2026-04-24 18:13:21
Current date and time (UTC):   2026-04-24 21:13:21

Current TinyTroupe configuration 
[OpenAI]
api_type = azure
azure_api_version = 2024-12-01-preview
model = gpt-5-mini
reasoning_model = o3-mini
vision_detail = auto
embedding_model = text-embedding-3-small
azure_embedding_model_api_version = 2023-05-15
max_completion_tokens = 128000
timeout = 300
max_attempts = 5
waiting_tim

In [2]:
#config_manager.update("loglevel", "WARNING")

## Parameters

In [3]:
full_mode = True  # set to True to run the full mode with all agents and tasks

# avoid displaying the communication, to make the output cleaner for eval
TinyPerson.communication_display = False

In [4]:
if full_mode:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 12
    qty_proposals = 4

else:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 4
    qty_proposals = 2


## Experiment setup

In [5]:
experiment_runner = InPlaceExperimentRunner("./brainstorming_and_focus_group_quantitative_experimentation_2.1.json")

experiment_runner.add_experiment("Control")
experiment_runner.add_experiment("Treatment")

In [6]:
experiment_runner.activate_next_experiment()

#experiment_runner.fix_active_experiment("Control")
#experiment_runner.fix_active_experiment("Treatment")

In [7]:
print(f"Running experiment {experiment_runner.get_active_experiment()}")

Running experiment Control


## Agents and populations

In [8]:

people = []
if not experiment_runner.has_finished_all_experiments():
    # load agents
    people = TinyPerson.load_specifications_from_folder("./population/difficult_people_2")

    # filter to make it go faster?
    if qty_agents is not None:
        people = people[:qty_agents]

    # customize and print minibios 
    for person in people:
        person.import_fragment("./fragments/difficult_person.agent.fragment.json")
        print(person.minibio(extended=False))


Alan Merrick is a 48 year old Administrative Officer (Benefits and Records), British, currently living in Manchester, United Kingdom.
Anthony Russo is a 42 year old Journeyman Electrician / Senior Field Technician, American, currently living in Cleveland, Ohio, USA.
Anya Calder-Mori is a 45 year old Freelance Graphic Designer, Conceptual Artist and Cultural Critic, British, currently living in Camberwell, London, UK.
Barbara Jean Pratt is a 68 year old Retiree (former assembly line worker / part-time volunteer at church thrift shop), American, currently living in Small town near Toledo, Ohio, USA.
Colin Arthur Matthews is a 42 year old Operations Manager (Mid-level), British, currently living in Manchester, UK.
Colin Murray is a 52 year old Benefits and Housing Support Officer, British, currently living in Salford, Greater Manchester, UK.
Connor Walsh is a 28 year old Senior Customer Service Associate / Shift Lead (Retail Grocery Chain), American, currently living in Cleveland, Ohio, U

In [9]:
len(people)

12

In [10]:
# divide people in several groups of 5
people_groups = []
for i in range(0, len(people), 4):
    people_groups.append(people[i:i+4]
    )

len(people_groups)

3

In [11]:
# The experiment refers to customers

if experiment_runner.get_active_experiment() == "Control":
    for person in people:
        person.action_generator.enable_reasoning_step = False
        person.action_generator.enable_quality_checks = False

elif experiment_runner.get_active_experiment() == "Treatment":    
    for person in people:
       person.action_generator.enable_reasoning_step = False
       person.action_generator.enable_quality_checks = True
       person.action_generator.max_attempts = 2
       person.action_generator.enable_regeneration = True
       person.action_generator.quality_threshold = 5

## Proposals

In [12]:
proposals = [
    {"theme": "Daily Life and Convenience",
     "objective": "Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions."},

    {"theme": "Personal Growth and Wellbeing",
     "objective": "Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection."},

    {"theme": "Discovery and Exploration",
     "objective": "Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self."},

    {"theme": "Productivity and Resourcefulness",
     "objective": "Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively."},

    {"theme": "Creativity and Expression",
     "objective": "Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance artistic skills, or enable new forms of storytelling and communication."}
]

if not full_mode:
    proposals = proposals[:qty_proposals]

In [13]:
# divide the proposals in exactly two groups (half/half)
proposals_groups = []
proposals_groups.append(proposals[:len(proposals)//2])
proposals_groups.append(proposals[len(proposals)//2:])

proposals_groups

[[{'theme': 'Daily Life and Convenience',
   'objective': 'Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.'},
  {'theme': 'Personal Growth and Wellbeing',
   'objective': 'Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection.'}],
 [{'theme': 'Discovery and Exploration',
   'objective': 'Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.'},
  {'theme': 'Productivity and Resourcefulness',
   'objective': 'Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively.'},
  {'theme': 'Creativity and Expression',
   'objective': 'Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance art

## Auxiliary functions

In [14]:
def brainstorming_battery(agents, proposals, interventions, agent_propositions, environment_propositions, 
                          repetitions = 5, simulation_steps=10): 
    
    agent_propositions_scores = {}
    environment_propositions_scores = {}

    experiments_count = 0
    total_expected_experiments = len(proposals) * repetitions #* len(agents)

    # loop over proposals and repetitions
    for proposal in proposals:

        objective = proposal["objective"]
        theme = proposal["theme"]

        for i in range(repetitions):
            print("\n############## STARTING A NEW RESEARCH SESSION #################")
            print(f"Overall experiment number: {experiments_count+1} / {total_expected_experiments}")
            print(f"Discussion objective: {objective}")
            print(f"Trial number: {i+1}")
            print(f"Agents: {agents}")

            # clear the episodic memory of all agents
            for person in agents:
                person.clear_episodic_memory()

            world = TinyWorld(agents=agents, interventions=interventions)
            
            # Participants introduce themselves
            world.broadcast(f"""
                Hello everyone! Let's start by introducing ourselves, and mentioning problems we face in our daily personal
                and professional lives related to the following theme: {theme}
                
                Please:
                  - present yourself and your background;
                  - present some key personal problems related to the theme;
                  - present some key problems related to the theme that you face in your work;
                  - present some key problems related to the theme that you see in your industry as a whole.
                  
                Don't discuss solutions yet, just the problems you face and see others facing.
                """)
            world.run(1)
            
            # now to the brainstorming session itself
            world.broadcast(f"""
                Folks, your mission is to brainstorm {objective}. 
                Please follow these guidelines:
                  - give a unique and informative name to each idea you propose, so that it is easy to refer to it. Say it like "Idea name: '<name of the idea>'".;
                  - explain why you think it is a good idea, and what problem it solves, and how you feel about it;
                  - your ideas should be new complete, self-contained, products or services, not features for other existing products or services;
                  - think of creative ideas that would somehow help you in both in your personal and professional lives.
                  - create as many different and unique ideas as you can during the brainstorming session. Each idea must be **completely** different from the others 
                    (either by yourself or by others), and not just a variation of an existing idea.
                  - you should criticize each other's ideas, in order to make sure they are as
                    good as possible, but no more than once per idea.
                  - you should also provide suggestions for improvement to each other's ideas, in order to make them as good as possible, 
                    but no more than once per idea.
                  - regardless of critique or complement, you **must** primarily propose new ideas quickly, 
                    not just build on existing ones. 
                  - propose one idea at a time, instead of proposing multiple ideas at once, to allow appropriate discussion.
                  - you should **not** propose ideas that are too similar to each other, or to the ones already proposed by others.
                  - before saying anything, THINK deeply about yourself, your beliefs, interests, needs, life, etc., to come up with ideas that are
                    truly unique and different from the ones already proposed by others.
                   
                Please start the discussion now.
                """)
            world.run(simulation_steps)

            # extract and count ideas
            rapporteur = agents[0]  # the first agent is the rapporteur
            rapporteur.listen_and_act("Can you please consolidate the ideas that the group came up with? Provide a lot of details on each idea, and complement anything missing.")
            ideas = ResultsExtractor().extract_results_from_agent(rapporteur, 
                                    extraction_objective="Consolidates the ideas that the group came up with, explaining each idea as an item of a list." \
                                                        "Add information about: what problem the idea solves; to which target audience it is meant." \
                                                        "how is it different from competing, existing, products.", 
                                    situation="A focus group to brainstorm new product ideas.",
                                    fields= ["name", "description", "problem", "target_audience", "competition_analysis"],
                                    fields_hints={"ideas": "must be the root of the resulting dictionary."},)
            pprint(ideas)
            if "ideas_qty" not in environment_propositions_scores:
                environment_propositions_scores["ideas_qty"] = []
            if ideas is not None and "ideas" in ideas and isinstance(ideas["ideas"], list):
                environment_propositions_scores["ideas_qty"].append(len(ideas["ideas"]))

            # Evaluate environment propositions in parallel
            env_results = parallel_map_dict(
                environment_propositions,
                lambda item: item[1].copy().score(
                    world, 
                    claim_variables={"task_description": f"A brainstorming or focus group session was run about: {objective}."}, 
                    return_full_response=True
                )
            )
            
            # Process environment results
            for k, result in env_results.items():
                if k not in environment_propositions_scores:
                    environment_propositions_scores[k] = []
                environment_propositions_scores[k].append(result["value"])
                print("value: ", result["value"])
                print("justification: ", result["justification"])
                print("reasoning: ", result["reasoning"])

            # Evaluate agent propositions across all agents in parallel
            agent_results = parallel_map_cross(
                [agents, agent_propositions.items()],
                lambda agent, prop_item: (
                    prop_item[0],  # proposition key
                    prop_item[1].copy().score(agent, return_full_response=True)  # result
                )
            )
            
            # Process agent results
            for k, result in agent_results:
                if k not in agent_propositions_scores:
                    agent_propositions_scores[k] = []
                if result is not None:
                    agent_propositions_scores[k].append(result["value"])
                    print("value: ", result["value"])
                    print("justification: ", result["justification"])
                    print("reasoning: ", result["reasoning"])
                    print("\n\n")
                else:
                    print(f"*****WARNING:***** Agent did not respond to proposition {k}.")
            #
            ##for k, proposition in agent_propositions.items():
            ##    for person in world.agents:
            ##        result = proposition.copy().score(person, return_full_response=True)
            ##        
            ##        if k not in agent_propositions_scores:
            ##            agent_propositions_scores[k] = []
            ##        agent_propositions_scores[k].append(result["value"])
            ##
            ##        print("value: ", result["value"])
            ##        print("justification: ", result["justification"])
            ##        print("reasoning: ", result["reasoning"])
            ##        print("\n\n")
            ##
            
            experiments_count += 1
            print("\n\n")

    return agent_propositions_scores, environment_propositions_scores



## Perform experiment

In [15]:
agent_propositions_scores={}
environment_propositions_scores={}

In [16]:
def brainstorm(people, proposals=proposals):
    global agent_propositions_scores, environment_propositions_scores
    if not experiment_runner.has_finished_all_experiments():

        interventions = []
        if experiment_runner.get_active_experiment() == "Treatment":
            interventions = \
                Intervention.create_for_each(people)\
                    .set_functional_precondition(lambda target: target.actions_count >=7)\
                    .set_textual_precondition(
                        """
                        AGENT IS NOT PROPOSING COMPLETELY NEW PRODUCT/SERVICE IDEAS ANYMORE:
                        The last **entirely** new product/service idea proposed by this agent, if any, was proposed by him/her **more** than 5 of simulation events ago.
                        That is to say, the agent has not proposed any new product/service idea in the last 5 of his/her simulation trajectory events.
                        Additional features, variations of or other refinements to product/service ideas already proposed are NOT considered new!

                        How to compute the steps gap:
                        1. Determine the current next event number (N); and the last event number in which the agent proposed a new product/service idea (M).
                            This information can be found in the simulation trajectory.
                        2. Compute the **difference** beteween the current next event number and the last event number in which the agent proposed a new product/service idea: D = N - M
                        3. The proposition is true if, and only if, the difference D is **greater than** 5.
                        """)\
                    .set_effect(lambda target: target.think("""
                                                            I need to propose additional, **completelly** new and different, product/service ideas. This was part of the requirement for this session.
                                                            I will propose an entirely **new** idea now, I **cannot** repeat or refine previous ideas! I cannot make variations
                                                            of previous ideas (e.g., "XYZ for A", "XYZ for B", "XYZ for Z" are repetitive, there should be only one "XYZ"), 
                                                            I need to think of something **entirely** new and different.
                                                            To help me avoid repeating previous ideas, I'll now explicitly THINK about all the ideas already given by myself or
                                                            others, and then, based on that, I'll think again about a new unique idea.
                                                            """))

                                                            
        tmp_agent_propositions_scores, tmp_environment_propositions_scores = \
            brainstorming_battery(
                agents=people,
                proposals=proposals,
                interventions=interventions,    
                agent_propositions={
                    "Hard Persona Adherence": hard_persona_adherence,
                    "Self-consistency": self_consistency,
                    "Fluency": fluency
                },
                environment_propositions={
                    "Task Completion": task_completion,
                    "Divergence": divergence
                },
                repetitions=repetitions_per_task,
                simulation_steps=simulation_steps
            )

        pprint("NEW AGENT PROPOSITIONS SCORES")
        pprint(tmp_agent_propositions_scores)
        print("\n\n")
        pprint("NEW ENVIRONMENT PROPOSITIONS SCORES")
        pprint(tmp_environment_propositions_scores)

        # merge the scores lists
        agent_propositions_scores = merge_dicts_of_lists(tmp_agent_propositions_scores, agent_propositions_scores)
        environment_propositions_scores = merge_dicts_of_lists(tmp_environment_propositions_scores, environment_propositions_scores)

        return agent_propositions_scores, environment_propositions_scores

In [17]:
brainstorm(people_groups[0], proposals_groups[0]) if len(people_groups) > 0  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Alan Merrick'), TinyPerson(name='Anthony Russo'), TinyPerson(name='Anya Calder-Mori'), TinyPerson(name='Barbara Jean Pratt')]
2026-04-24 18:15:00,357 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 1] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 1 step 1 of 1 ─────────────────────────────────────────────

2026-04-24 18:15:00,394 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-04-24 18:15:05,594 - ThreadPoolExecutor-0_0(45840) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 18:15:05,611 - ThreadPoolExecutor-0_3(42252) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 18:15:05,685 - ThreadPoolExecutor-0_1(33384) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 18:15:05,759 - ThreadPoolExecutor-0_2(46792) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 18:15:06,542 - ThreadPoolExecutor-0_2(46792) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 18:15:06,545 - ThreadPoolExecutor-0_3(42252) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 18:15:06,550 - ThreadPoolExecutor-0_1(33384) - tinytroupe - INFO - Waiting

───────────────────────────────────────────── TinyWorld 1 step 1 of 5 ─────────────────────────────────────────────

2026-04-24 18:15:59,738 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-04-24 18:16:03,427 - ThreadPoolExecutor-1_3(46816) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 18:16:03,454 - ThreadPoolExecutor-1_2(7844) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 18:16:03,604 - ThreadPoolExecutor-1_2(7844) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 18:16:03,609 - ThreadPoolExecutor-1_3(46816) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 18:16:03,944 - ThreadPoolExecutor-1_0(40980) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 18:16:03,970 - ThreadPoolExecutor-1_1(36304) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 18:16:04,083 - ThreadPoolExecutor-1_1(36304) - tinytroupe - INFO - Waiting 5

...

───────────────────────────────────────────── TinyWorld 4 step 4 of 5 ─────────────────────────────────────────────

2026-04-24 19:09:08,232 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 4] No timedelta provided, so the datetime was not advanced.
2026-04-24 19:09:10,720 - ThreadPoolExecutor-28_0(15548) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:09:10,773 - ThreadPoolExecutor-28_3(44564) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:09:10,780 - ThreadPoolExecutor-28_1(36612) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:09:10,780 - ThreadPoolExecutor-28_2(46712) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:09:10,826 - ThreadPoolExecutor-28_0(15548) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 19:09:10,871 - ThreadPoolExecutor-28_3(44564) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 19:09:10,892 - ThreadPoolExecutor-28_1(36612) - tinytroupe - INFO - 

───────────────────────────────────────────── TinyWorld 4 step 5 of 5 ─────────────────────────────────────────────

2026-04-24 19:10:08,661 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 4] No timedelta provided, so the datetime was not advanced.
2026-04-24 19:10:10,664 - ThreadPoolExecutor-29_2(39716) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:10:10,678 - ThreadPoolExecutor-29_0(30392) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:10:10,685 - ThreadPoolExecutor-29_3(43336) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:10:10,700 - ThreadPoolExecutor-29_1(13036) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:10:10,751 - ThreadPoolExecutor-29_2(39716) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 19:10:10,770 - ThreadPoolExecutor-29_0(30392) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 19:10:10,798 - ThreadPoolExecutor-29_3(43336) - tinytroupe - INFO - 

({'Hard Persona Adherence': [0, 0, 2, 0, 0, 1, 4, 3, 3, 1, 0, 4, 2, 5, 0, 0],
  'Self-consistency': [9, 9, 9, 9, 9, 9, 9, 9, 3, 9, 9, 9, 9, 9, 2, 9],
  'Fluency': [7, 7, 9, 8, 7, 6, 9, 9, 9, 6, 8, 9, 8, 8, 9, 8]},
 {'ideas_qty': [4, 4, 4, 4],
  'Task Completion': [9, 9, 9, 9],
  'Divergence': [2, 0, 0, 1]})

In [18]:
brainstorm(people_groups[0], proposals_groups[1]) if len(people_groups) > 0  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Alan Merrick'), TinyPerson(name='Anthony Russo'), TinyPerson(name='Anya Calder-Mori'), TinyPerson(name='Barbara Jean Pratt')]
2026-04-24 19:20:24,455 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 5] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 5 step 1 of 1 ─────────────────────────────────────────────

2026-04-24 19:20:24,467 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 5] No timedelta provided, so the datetime was not advanced.
2026-04-24 19:20:27,952 - ThreadPoolExecutor-32_2(28480) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:20:27,960 - ThreadPoolExecutor-32_3(41684) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:20:28,034 - ThreadPoolExecutor-32_2(28480) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 19:20:28,047 - ThreadPoolExecutor-32_3(41684) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 19:20:28,140 - ThreadPoolExecutor-32_0(36640) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:20:28,150 - ThreadPoolExecutor-32_1(21348) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:20:28,216 - ThreadPoolExecutor-32_0(36640) - tinytroupe - INFO - 

───────────────────────────────────────────── TinyWorld 5 step 1 of 5 ─────────────────────────────────────────────

2026-04-24 19:21:33,038 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 5] No timedelta provided, so the datetime was not advanced.
2026-04-24 19:21:38,438 - ThreadPoolExecutor-33_2(8956) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:21:38,463 - ThreadPoolExecutor-33_3(33800) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:21:38,558 - ThreadPoolExecutor-33_2(8956) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 19:21:38,575 - ThreadPoolExecutor-33_3(33800) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 19:21:38,626 - ThreadPoolExecutor-33_0(17856) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:21:38,665 - ThreadPoolExecutor-33_1(7712) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 19:21:38,705 - ThreadPoolExecutor-33_0(17856) - tinytroupe - INFO - Wai

...

──────────────────────────────────────────── TinyWorld 10 step 4 of 5 ─────────────────────────────────────────────

2026-04-24 20:40:12,307 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 10] No timedelta provided, so the datetime was not advanced.
2026-04-24 20:40:14,524 - ThreadPoolExecutor-76_0(44784) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:40:14,539 - ThreadPoolExecutor-76_1(48508) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:40:14,546 - ThreadPoolExecutor-76_2(1900) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:40:14,571 - ThreadPoolExecutor-76_3(42516) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:40:14,618 - ThreadPoolExecutor-76_0(44784) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 20:40:14,647 - ThreadPoolExecutor-76_1(48508) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 20:40:14,662 - ThreadPoolExecutor-76_2(1900) - tinytroupe - INFO - W

──────────────────────────────────────────── TinyWorld 10 step 5 of 5 ─────────────────────────────────────────────

2026-04-24 20:41:02,378 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 10] No timedelta provided, so the datetime was not advanced.
2026-04-24 20:41:05,192 - ThreadPoolExecutor-77_3(33592) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:41:05,263 - ThreadPoolExecutor-77_0(44092) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:41:05,300 - ThreadPoolExecutor-77_3(33592) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 20:41:05,355 - ThreadPoolExecutor-77_0(44092) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 20:41:05,674 - ThreadPoolExecutor-77_2(7528) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:41:05,713 - ThreadPoolExecutor-77_1(33416) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:41:05,754 - ThreadPoolExecutor-77_2(7528) - tinytroupe - INFO - W

({'Hard Persona Adherence': [0,
   0,
   2,
   0,
   0,
   1,
   4,
   3,
   3,
   1,
   0,
   4,
   2,
   5,
   0,
   0,
   3,
   1,
   2,
   0,
   0,
   5,
   3,
   2,
   3,
   1,
   0,
   1,
   3,
   5,
   3,
   0,
   1,
   2,
   1,
   5,
   2,
   2,
   2,
   3],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   3,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9],
  'Fluency': [7,
   7,
   9,
   8,
   7,
   6,
   9,
   9,
   9,
   6,
   8,
   9,
   8,
   8,
   9,
   8,
   8,
   9,
   8,
   9,
   6,
   8,
   8,
   9,
   8,
   9,
   8,
   8,
   6,
   8,
   9,
   9,
   7,
   8,
   9,
   9,
   9,
   9,
   8,
   8]},
 {'ideas_qty': [4, 4, 4, 4, 4, 4, 4, 5, 4, 4],
  'Task Completion': [9, 9, 9, 9, 9, 9, 9, 9, 9, 9],
  'Divergence': [2, 0, 0, 1, 0, 0, 0, 1, 1, 0]})

In [19]:
brainstorm(people_groups[1], proposals_groups[0]) if len(people_groups) > 1  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Colin Arthur Matthews'), TinyPerson(name='Colin Murray'), TinyPerson(name='Connor Walsh'), TinyPerson(name='Darren McCall')]
2026-04-24 20:50:04,850 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 11] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 11 step 1 of 1 ─────────────────────────────────────────────

2026-04-24 20:50:04,856 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-04-24 20:50:07,218 - ThreadPoolExecutor-80_1(22596) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:50:07,273 - ThreadPoolExecutor-80_1(22596) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 20:50:07,308 - ThreadPoolExecutor-80_0(41240) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:50:07,314 - ThreadPoolExecutor-80_2(48408) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:50:07,315 - ThreadPoolExecutor-80_3(31276) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:50:07,407 - ThreadPoolExecutor-80_0(41240) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 20:50:07,415 - ThreadPoolExecutor-80_2(48408) - tinytroupe - INFO -

──────────────────────────────────────────── TinyWorld 11 step 1 of 5 ─────────────────────────────────────────────

2026-04-24 20:50:55,892 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-04-24 20:50:59,565 - ThreadPoolExecutor-81_2(48376) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:50:59,591 - ThreadPoolExecutor-81_1(34864) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:50:59,641 - ThreadPoolExecutor-81_2(48376) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 20:50:59,671 - ThreadPoolExecutor-81_1(34864) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 20:51:00,059 - ThreadPoolExecutor-81_0(43592) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 20:51:00,170 - ThreadPoolExecutor-81_0(43592) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 20:51:00,190 - ThreadPoolExecutor-81_3(44

...

──────────────────────────────────────────── TinyWorld 14 step 4 of 5 ─────────────────────────────────────────────

2026-04-24 21:36:56,247 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 14] No timedelta provided, so the datetime was not advanced.
2026-04-24 21:36:58,653 - ThreadPoolExecutor-108_2(43584) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:36:58,678 - ThreadPoolExecutor-108_3(45504) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:36:58,727 - ThreadPoolExecutor-108_2(43584) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 21:36:58,768 - ThreadPoolExecutor-108_3(45504) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 21:36:58,808 - ThreadPoolExecutor-108_1(43884) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:36:58,878 - ThreadPoolExecutor-108_1(43884) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 21:36:58,897 - ThreadPoolExecutor-1

──────────────────────────────────────────── TinyWorld 14 step 5 of 5 ─────────────────────────────────────────────

2026-04-24 21:37:44,736 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 14] No timedelta provided, so the datetime was not advanced.
2026-04-24 21:37:47,474 - ThreadPoolExecutor-109_3(26400) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:37:47,484 - ThreadPoolExecutor-109_2(45716) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:37:47,590 - ThreadPoolExecutor-109_2(45716) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 21:37:47,595 - ThreadPoolExecutor-109_3(26400) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 21:37:47,644 - ThreadPoolExecutor-109_1(47560) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:37:47,683 - ThreadPoolExecutor-109_0(42968) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:37:47,729 - ThreadPoolExecutor-109_1(47560) - tinytroupe -

({'Hard Persona Adherence': [0,
   0,
   2,
   0,
   0,
   1,
   4,
   3,
   3,
   1,
   0,
   4,
   2,
   5,
   0,
   0,
   3,
   1,
   2,
   0,
   0,
   5,
   3,
   2,
   3,
   1,
   0,
   1,
   3,
   5,
   3,
   0,
   1,
   2,
   1,
   5,
   2,
   2,
   2,
   3,
   1,
   0,
   3,
   3,
   0,
   0,
   2,
   1,
   3,
   0,
   5,
   3,
   3,
   3,
   2,
   0],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   3,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9],
  'Fluency': [7,
   7,
   9,
   8,
   7,
   6,
   9,
   9,
   9,
   6,
   8,
   9,
   8,
   8,
   9,
   8,
   8,
   9,
   8,
   9,
   6,
   8,
   8,
   9,
   8,
   9,
   8,
   8,
   6,
   8,
   9,
   9,
   7,
   8,
   9,
   9,
   9,
   9,
   8,
   8,
   7,
   7,
   8,
   9,
   8,

In [20]:
brainstorm(people_groups[1], proposals_groups[1]) if len(people_groups) > 1  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Colin Arthur Matthews'), TinyPerson(name='Colin Murray'), TinyPerson(name='Connor Walsh'), TinyPerson(name='Darren McCall')]
2026-04-24 21:46:41,542 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 15] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 15 step 1 of 1 ─────────────────────────────────────────────

2026-04-24 21:46:41,548 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 15] No timedelta provided, so the datetime was not advanced.
2026-04-24 21:46:44,148 - ThreadPoolExecutor-112_1(13332) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:46:44,156 - ThreadPoolExecutor-112_0(20140) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:46:44,231 - ThreadPoolExecutor-112_1(13332) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 21:46:44,240 - ThreadPoolExecutor-112_0(20140) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 21:46:44,536 - ThreadPoolExecutor-112_3(27128) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:46:44,557 - ThreadPoolExecutor-112_2(28180) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:46:44,617 - ThreadPoolExecutor-112_3(27128) - tinytroupe -

──────────────────────────────────────────── TinyWorld 15 step 1 of 5 ─────────────────────────────────────────────

2026-04-24 21:47:42,259 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 15] No timedelta provided, so the datetime was not advanced.
2026-04-24 21:47:43,967 - ThreadPoolExecutor-113_3(47084) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:47:44,011 - ThreadPoolExecutor-113_1(47712) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:47:44,017 - ThreadPoolExecutor-113_0(31052) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:47:44,032 - ThreadPoolExecutor-113_2(12204) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 21:47:44,041 - ThreadPoolExecutor-113_3(47084) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 21:47:44,079 - ThreadPoolExecutor-113_1(47712) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 21:47:44,107 - ThreadPoolExecutor-113_0(31052) - tinytroupe -

...

──────────────────────────────────────────── TinyWorld 20 step 4 of 5 ─────────────────────────────────────────────

2026-04-24 23:07:40,849 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 20] No timedelta provided, so the datetime was not advanced.
2026-04-24 23:07:42,560 - ThreadPoolExecutor-156_0(34344) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:07:42,587 - ThreadPoolExecutor-156_1(48492) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:07:42,599 - ThreadPoolExecutor-156_3(36372) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:07:42,605 - ThreadPoolExecutor-156_2(42460) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:07:42,635 - ThreadPoolExecutor-156_0(34344) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 23:07:42,645 - ThreadPoolExecutor-156_1(48492) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 23:07:42,663 - ThreadPoolExecutor-156_3(36372) - tinytroupe -

──────────────────────────────────────────── TinyWorld 20 step 5 of 5 ─────────────────────────────────────────────

2026-04-24 23:08:36,239 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 20] No timedelta provided, so the datetime was not advanced.
2026-04-24 23:08:38,410 - ThreadPoolExecutor-157_3(33336) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:08:38,469 - ThreadPoolExecutor-157_1(44808) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:08:38,481 - ThreadPoolExecutor-157_2(36152) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:08:38,503 - ThreadPoolExecutor-157_0(31152) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:08:38,532 - ThreadPoolExecutor-157_3(33336) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 23:08:38,565 - ThreadPoolExecutor-157_1(44808) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 23:08:38,599 - ThreadPoolExecutor-157_0(31152) - tinytroupe -

({'Hard Persona Adherence': [0,
   0,
   2,
   0,
   0,
   1,
   4,
   3,
   3,
   1,
   0,
   4,
   2,
   5,
   0,
   0,
   3,
   1,
   2,
   0,
   0,
   5,
   3,
   2,
   3,
   1,
   0,
   1,
   3,
   5,
   3,
   0,
   1,
   2,
   1,
   5,
   2,
   2,
   2,
   3,
   1,
   0,
   3,
   3,
   0,
   0,
   2,
   1,
   3,
   0,
   5,
   3,
   3,
   3,
   2,
   0,
   3,
   3,
   0,
   0,
   2,
   0,
   0,
   0,
   2,
   1,
   3,
   2,
   2,
   2,
   0,
   4,
   0,
   0,
   0,
   2,
   0,
   0,
   3,
   1],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   3,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   7,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,

In [21]:
brainstorm(people_groups[2], proposals_groups[0]) if len(people_groups) > 2  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Dean Bartlett'), TinyPerson(name='Declan Blackwell'), TinyPerson(name='Edgar Milton Crane'), TinyPerson(name='Leonard Victor Hale')]
2026-04-24 23:18:48,417 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 21] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 21 step 1 of 1 ─────────────────────────────────────────────

2026-04-24 23:18:48,423 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-04-24 23:18:50,132 - ThreadPoolExecutor-160_0(43516) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:18:50,152 - ThreadPoolExecutor-160_1(31152) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:18:50,157 - ThreadPoolExecutor-160_3(14772) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:18:50,178 - ThreadPoolExecutor-160_2(15068) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:18:50,189 - ThreadPoolExecutor-160_0(43516) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 23:18:50,208 - ThreadPoolExecutor-160_1(31152) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 23:18:50,224 - ThreadPoolExecutor-160_3(14772) - tinytroupe -

──────────────────────────────────────────── TinyWorld 21 step 1 of 5 ─────────────────────────────────────────────

2026-04-24 23:19:49,189 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-04-24 23:19:52,674 - ThreadPoolExecutor-161_3(41432) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:19:52,741 - ThreadPoolExecutor-161_3(41432) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 23:19:52,749 - ThreadPoolExecutor-161_0(39156) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:19:52,829 - ThreadPoolExecutor-161_0(39156) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-24 23:19:53,022 - ThreadPoolExecutor-161_1(32316) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:19:53,074 - ThreadPoolExecutor-161_2(49140) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-24 23:19:53,110 - ThreadPoolExecutor-161_1(32316) - tinytroupe -

...

──────────────────────────────────────────── TinyWorld 24 step 4 of 5 ─────────────────────────────────────────────

2026-04-25 00:05:39,204 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 24] No timedelta provided, so the datetime was not advanced.
2026-04-25 00:05:40,707 - ThreadPoolExecutor-188_3(41464) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:05:40,727 - ThreadPoolExecutor-188_1(42396) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:05:40,731 - ThreadPoolExecutor-188_0(41588) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:05:40,732 - ThreadPoolExecutor-188_2(48720) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:05:40,764 - ThreadPoolExecutor-188_3(41464) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 00:05:40,781 - ThreadPoolExecutor-188_1(42396) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 00:05:40,798 - ThreadPoolExecutor-188_0(41588) - tinytroupe -

──────────────────────────────────────────── TinyWorld 24 step 5 of 5 ─────────────────────────────────────────────

2026-04-25 00:06:24,481 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 24] No timedelta provided, so the datetime was not advanced.
2026-04-25 00:06:26,179 - ThreadPoolExecutor-189_0(4980) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:06:26,193 - ThreadPoolExecutor-189_2(48360) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:06:26,207 - ThreadPoolExecutor-189_3(33964) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:06:26,213 - ThreadPoolExecutor-189_1(48944) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:06:26,246 - ThreadPoolExecutor-189_0(4980) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 00:06:26,271 - ThreadPoolExecutor-189_1(48944) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 00:06:26,278 - ThreadPoolExecutor-189_3(33964) - tinytroupe - I

({'Hard Persona Adherence': [0,
   0,
   2,
   0,
   0,
   1,
   4,
   3,
   3,
   1,
   0,
   4,
   2,
   5,
   0,
   0,
   3,
   1,
   2,
   0,
   0,
   5,
   3,
   2,
   3,
   1,
   0,
   1,
   3,
   5,
   3,
   0,
   1,
   2,
   1,
   5,
   2,
   2,
   2,
   3,
   1,
   0,
   3,
   3,
   0,
   0,
   2,
   1,
   3,
   0,
   5,
   3,
   3,
   3,
   2,
   0,
   3,
   3,
   0,
   0,
   2,
   0,
   0,
   0,
   2,
   1,
   3,
   2,
   2,
   2,
   0,
   4,
   0,
   0,
   0,
   2,
   0,
   0,
   3,
   1,
   2,
   1,
   3,
   1,
   1,
   1,
   2,
   1,
   2,
   1,
   2,
   0,
   1,
   4,
   3,
   0],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   3,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   7,
   8,
   9,
   9,
   9,
   9,
   9,

In [22]:
brainstorm(people_groups[2], proposals_groups[1]) if len(people_groups) > 2  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Dean Bartlett'), TinyPerson(name='Declan Blackwell'), TinyPerson(name='Edgar Milton Crane'), TinyPerson(name='Leonard Victor Hale')]
2026-04-25 00:14:49,605 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 25] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 25 step 1 of 1 ─────────────────────────────────────────────

2026-04-25 00:14:49,611 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 25] No timedelta provided, so the datetime was not advanced.
2026-04-25 00:14:51,817 - ThreadPoolExecutor-192_3(41356) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:14:51,866 - ThreadPoolExecutor-192_3(41356) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 00:14:51,867 - ThreadPoolExecutor-192_0(34748) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:14:51,912 - ThreadPoolExecutor-192_1(47228) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:14:51,917 - ThreadPoolExecutor-192_2(32228) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:14:51,943 - ThreadPoolExecutor-192_0(34748) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 00:14:51,966 - ThreadPoolExecutor-192_1(47228) - tinytroupe -

──────────────────────────────────────────── TinyWorld 25 step 1 of 5 ─────────────────────────────────────────────

2026-04-25 00:15:43,755 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 25] No timedelta provided, so the datetime was not advanced.
2026-04-25 00:15:46,075 - ThreadPoolExecutor-193_2(44148) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:15:46,094 - ThreadPoolExecutor-193_0(45484) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:15:46,117 - ThreadPoolExecutor-193_3(15424) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:15:46,124 - ThreadPoolExecutor-193_1(42816) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 00:15:46,170 - ThreadPoolExecutor-193_2(44148) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 00:15:46,190 - ThreadPoolExecutor-193_0(45484) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 00:15:46,207 - ThreadPoolExecutor-193_3(15424) - tinytroupe -

...

──────────────────────────────────────────── TinyWorld 30 step 4 of 5 ─────────────────────────────────────────────

2026-04-25 01:32:49,284 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 30] No timedelta provided, so the datetime was not advanced.
2026-04-25 01:32:51,249 - ThreadPoolExecutor-236_0(37824) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 01:32:51,309 - ThreadPoolExecutor-236_1(46688) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 01:32:51,325 - ThreadPoolExecutor-236_2(43888) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 01:32:51,340 - ThreadPoolExecutor-236_3(26140) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 01:32:51,354 - ThreadPoolExecutor-236_0(37824) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 01:32:51,398 - ThreadPoolExecutor-236_1(46688) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 01:32:51,418 - ThreadPoolExecutor-236_2(43888) - tinytroupe -

──────────────────────────────────────────── TinyWorld 30 step 5 of 5 ─────────────────────────────────────────────

2026-04-25 01:33:46,992 - MainThread(29564) - tinytroupe - INFO - [TinyWorld 30] No timedelta provided, so the datetime was not advanced.
2026-04-25 01:33:48,714 - ThreadPoolExecutor-237_1(33676) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 01:33:48,719 - ThreadPoolExecutor-237_0(4372) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 01:33:48,720 - ThreadPoolExecutor-237_2(48712) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 01:33:48,731 - ThreadPoolExecutor-237_3(11988) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-25 01:33:48,776 - ThreadPoolExecutor-237_1(33676) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 01:33:48,787 - ThreadPoolExecutor-237_0(4372) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-25 01:33:48,805 - ThreadPoolExecutor-237_3(11988) - tinytroupe - I

({'Hard Persona Adherence': [0,
   0,
   2,
   0,
   0,
   1,
   4,
   3,
   3,
   1,
   0,
   4,
   2,
   5,
   0,
   0,
   3,
   1,
   2,
   0,
   0,
   5,
   3,
   2,
   3,
   1,
   0,
   1,
   3,
   5,
   3,
   0,
   1,
   2,
   1,
   5,
   2,
   2,
   2,
   3,
   1,
   0,
   3,
   3,
   0,
   0,
   2,
   1,
   3,
   0,
   5,
   3,
   3,
   3,
   2,
   0,
   3,
   3,
   0,
   0,
   2,
   0,
   0,
   0,
   2,
   1,
   3,
   2,
   2,
   2,
   0,
   4,
   0,
   0,
   0,
   2,
   0,
   0,
   3,
   1,
   2,
   1,
   3,
   1,
   1,
   1,
   2,
   1,
   2,
   1,
   2,
   0,
   1,
   4,
   3,
   0,
   5,
   3,
   0,
   1,
   2,
   1,
   0,
   3,
   2,
   0,
   2,
   3,
   1,
   9,
   0,
   3,
   2,
   1,
   4,
   1,
   3,
   1,
   2,
   3],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   3,
   9,
   9,
   9,
   9,
   9,
   2,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,

In [23]:
brainstorm(people_groups[3], proposals_groups[0]) if len(people_groups) > 3  and len(proposals_groups) > 0 else None

In [24]:
brainstorm(people_groups[3], proposals_groups[1]) if len(people_groups) > 3  and len(proposals_groups) > 1 else None

In [25]:
brainstorm(people_groups[4], proposals_groups[0]) if len(people_groups) > 4  and len(proposals_groups) > 0 else None

In [26]:
brainstorm(people_groups[4], proposals_groups[1]) if len(people_groups) > 4  and len(proposals_groups) > 1 else None

## Extract results and analyze

In [27]:
if experiment_runner.get_active_experiment() in ["Control", "Treatment"]:
    combined_scores = {**agent_propositions_scores, **environment_propositions_scores}
    experiment_runner.add_experiment_results(combined_scores, experiment_name=experiment_runner.get_active_experiment()) 
    
    plot_scores(combined_scores)

else:
    print("Experiment finished. No more experiments to run.")

{'Divergence': [2,
                0,
                0,
                1,
                0,
                0,
                0,
                1,
                1,
                0,
                3,
                0,
                0,
                0,
                0,
                2,
                0,
                5,
                0,
                0,
                1,
                0,
                0,
                0,
                0,
                0,
                0,
                0,
                1,
                1],
 'Fluency': [7,
             7,
             9,
             8,
             7,
             6,
             9,
             9,
             9,
             6,
             8,
             9,
             8,
             8,
             9,
             8,
             8,
             9,
             8,
             9,
             6,
             8,
             8,
             9,
             8,
             9,
             

,Proposition,Average Score,Standard Deviation,Count
0,Hard Persona Adherence,1.758333,1.582445,120.0
1,Self-consistency,8.683333,1.263471,120.0
2,Fluency,8.025000,0.727174,120.0
3,ideas_qty,3.793103,0.559292,29.0
4,Task Completion,8.966667,0.182574,30.0
5,Divergence,0.600000,1.132589,30.0


In [28]:
if experiment_runner.has_finished_all_experiments():
    print("All experiments have been finished.")
    print(f"STATISTICTS: Control vs")
    pprint(experiment_runner.run_statistical_tests(control_experiment_name='Control'))

    # plot scores of both experiments
    experiment_control_scores = experiment_runner.get_experiment_results("Control")
    experiment_treatment_scores = experiment_runner.get_experiment_results("Treatment")
    
    
    plot_scores(experiment_control_scores)
    plot_scores(experiment_treatment_scores)

else:
    print("Not all experiments have been finished. RESTART AND RERUN.")

Not all experiments have been finished. RESTART AND RERUN.


In [29]:
experiment_runner.finish_active_experiment()

2026-04-25 01:42:52,779 - MainThread(29564) - tinytroupe - INFO - Experiment 'Control' marked as finished.


True